# LSTM

## Imports

In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import json 

## Preprocessing

In [3]:
from Load_dataset import load_datasets, load_test_datasets
# Define the base path for the dataset
base_path = 'datasets/serveDataset/'  # Set the correct path

# Load datasets
images, labels, keypoints = load_datasets(base_path)
test_images, test_labels, test_keypoints = load_test_datasets(base_path)

# Convert keypoints to LSTM-compatible format
# Assuming keypoints shape is (num_samples, num_timesteps, num_features)
def reshape_keypoints_for_lstm(keypoints):
    return keypoints.reshape(keypoints.shape[0], keypoints.shape[1], -1)  # Reshape to (N, T, F)

train_keypoints_lstm = reshape_keypoints_for_lstm(keypoints)
test_keypoints_lstm = reshape_keypoints_for_lstm(test_keypoints)

# Create DataLoaders
train_dataset = TensorDataset(torch.tensor(train_keypoints_lstm, dtype=torch.float32), torch.tensor(labels, dtype=torch.float32))
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

test_dataset = TensorDataset(torch.tensor(test_keypoints_lstm, dtype=torch.float32), torch.tensor(test_labels, dtype=torch.float32))
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

## Model Training

### Model creation + tuning

In [5]:
# model creation

class KeypointLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(KeypointLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)  # Assuming binary classification

    def forward(self, x):
        out, _ = self.lstm(x)  # Get LSTM outputs
        out = out[:, -1, :]  # Take the output of the last time step
        out = self.fc(out)

        print(f"LSTM Output shape: {out.shape}")  # Debugging: Check output shape
        return out  # Remove squeeze() for debugging


# Instantiate the model, loss function, and optimizer
input_size = train_keypoints_lstm.shape[2]  # Number of features (e.g., 14 keypoints)
hidden_size = 64
num_layers = 2

model = KeypointLSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers)

criterion = nn.BCEWithLogitsLoss()  # Binary Cross-Entropy Loss
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [6]:
def tune_hyperparameters(lr_values, hidden_sizes, num_layers_list, train_loader, val_loader, epochs=5, patience=2):
    best_accuracy = 0
    best_params = {}
    
    for lr in lr_values:
        for hidden_size in hidden_sizes:
            for num_layers in num_layers_list:
                print(f"Training with lr={lr}, hidden_size={hidden_size}, num_layers={num_layers}")

                # Instantiate the model, loss function, and optimizer
                model = KeypointLSTM(input_size=train_keypoints_lstm.shape[2], hidden_size=hidden_size, num_layers=num_layers)
                criterion = nn.BCEWithLogitsLoss()
                optimizer = torch.optim.Adam(model.parameters(), lr=lr)

                # Early stopping variables
                best_val_loss = float('inf')
                epochs_without_improvement = 0

                for epoch in range(epochs):
                    train_accuracy, train_loss = train(model, train_loader, optimizer, criterion, epochs=1)

                    # Validate the model
                    model.eval()
                    val_loss = 0
                    correct = 0
                    total = 0
                    with torch.no_grad():
                        for keypoints, labels in val_loader:
                            outputs = model(keypoints)
                            loss = criterion(outputs, labels.view(-1, 1))
                            val_loss += loss.item()
                            predictions = torch.sigmoid(outputs) > 0.5
                            correct += (predictions == labels.view(-1, 1)).sum().item()
                            total += labels.size(0)

                    avg_val_loss = val_loss / len(val_loader)
                    avg_val_accuracy = correct / total
                    print(f"Validation Epoch [{epoch+1}/{epochs}], Loss: {avg_val_loss:.4f}, Accuracy: {avg_val_accuracy:.4f}")

                    # Early stopping
                    if avg_val_loss < best_val_loss:
                        best_val_loss = avg_val_loss
                        epochs_without_improvement = 0  # Reset
                    else:
                        epochs_without_improvement += 1
                    
                    # Check for early stopping
                    if epochs_without_improvement >= patience:
                        print("Early stopping triggered.")
                        break

                # Check if the validation accuracy is the best
                if avg_val_accuracy > best_accuracy:
                    best_accuracy = avg_val_accuracy
                    best_params = {
                        "lr": lr,
                        "hidden_size": hidden_size,
                        "num_layers": num_layers
                    }

    return best_params

# Define hyperparameter search space
learning_rates = [0.001, 0.005]  # Reduced options to avoid excessive computation
hidden_sizes = [32, 64]  # Fewer hidden sizes for efficiency
num_layers = [1, 2]  # Fewer layers to reduce complexity

# Create validation DataLoader (you can split the train dataset into train and validation)
val_size = int(0.2 * len(train_dataset))  # 20% for validation
train_size = len(train_dataset) - val_size
train_subset, val_subset = torch.utils.data.random_split(train_dataset, [train_size, val_size])

train_loader = DataLoader(train_subset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=16, shuffle=False)

# Perform hyperparameter tuning
best_params = tune_hyperparameters(learning_rates, hidden_sizes, num_layers, train_loader, val_loader, epochs=5, patience=2)
print("Best hyperparameters:", best_params)

Training with lr=0.001, hidden_size=32, num_layers=1


  0%|          | 0/72 [00:00<?, ?it/s]

LSTM Output shape: torch.Size([16, 1])


 43%|████▎     | 31/72 [00:00<00:00, 128.97it/s]

LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.

...

LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])


 74%|███████▎  | 53/72 [00:00<00:00, 186.97it/s]

LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])


100%|██████████| 72/72 [00:00<00:00, 166.90it/s]

LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([6, 1])
Epoch [1/1], Loss: 0.6089, Accuracy: 0.7084
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([13, 1])
Validation Epoch [3/5

In [9]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import json
import optuna

# Define the LSTM model
class KeypointLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(KeypointLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)  # Assuming binary classification

    def forward(self, x):
        out, _ = self.lstm(x)  # Get LSTM outputs
        out = out[:, -1, :]  # Take the output of the last time step
        out = self.fc(out)
        return out  # Do not apply squeeze for debugging

# Function to train the model
def train(model, loader, optimizer, criterion, epochs=1):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        total = 0

        for keypoints, labels in tqdm(loader):
            optimizer.zero_grad()
            outputs = model(keypoints)  # Outputs shape should be [N, 1]
            loss = criterion(outputs, labels.view(-1, 1))  # Ensure labels are shaped [N, 1]
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            predictions = torch.sigmoid(outputs) > 0.5  # Convert to binary predictions
            correct += (predictions == labels.view(-1, 1)).sum().item()  # Ensure labels are shaped [N, 1]
            total += labels.size(0)

        avg_loss = total_loss / len(loader)
        avg_accuracy = correct / total
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}, Accuracy: {avg_accuracy:.4f}")

    return avg_accuracy, avg_loss

# Define the objective function for Optuna
def objective(trial):
    # Hyperparameters to tune
    lr = trial.suggest_loguniform('lr', 1e-5, 1e-1)  # Learning rate
    hidden_size = trial.suggest_categorical('hidden_size', [32, 64, 128])  # Hidden size
    num_layers = trial.suggest_int('num_layers', 1, 3)  # Number of LSTM layers

    # Instantiate the model, loss function, and optimizer
    model = KeypointLSTM(input_size=train_keypoints_lstm.shape[2], hidden_size=hidden_size, num_layers=num_layers)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Create DataLoaders for training and validation
    train_loader = DataLoader(train_subset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=16, shuffle=False)

    # Early stopping variables
    best_val_loss = float('inf')
    epochs_without_improvement = 0
    patience = 2

    for epoch in range(10):  # Train for a fixed number of epochs
        train_accuracy, train_loss = train(model, train_loader, optimizer, criterion, epochs=1)

        # Validate the model
        model.eval()
        val_loss = 0
        correct = 0
        total = 0
        with torch.no_grad():
            for keypoints, labels in val_loader:
                outputs = model(keypoints)
                loss = criterion(outputs, labels.view(-1, 1))
                val_loss += loss.item()
                predictions = torch.sigmoid(outputs) > 0.5
                correct += (predictions == labels.view(-1, 1)).sum().item()
                total += labels.size(0)

        avg_val_loss = val_loss / len(val_loader)
        avg_val_accuracy = correct / total
        print(f"Validation Epoch [{epoch+1}/10], Loss: {avg_val_loss:.4f}, Accuracy: {avg_val_accuracy:.4f}")

        # Check for early stopping
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
        
        if epochs_without_improvement >= patience:
            print("Early stopping triggered.")
            break

    return avg_val_accuracy  # Return validation accuracy for Optuna to optimize

# Load datasets and split into train/validation
base_path = 'datasets/serveDataset/'  # Set the correct path
images, labels, keypoints = load_datasets(base_path)
train_keypoints_lstm = reshape_keypoints_for_lstm(keypoints)

# Create TensorDataset and split for training and validation
train_dataset = TensorDataset(torch.tensor(train_keypoints_lstm, dtype=torch.float32), torch.tensor(labels, dtype=torch.float32))
val_size = int(0.2 * len(train_dataset))  # 20% for validation
train_size = len(train_dataset) - val_size
train_subset, val_subset = torch.utils.data.random_split(train_dataset, [train_size, val_size])

# Start the Optuna study
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)  # Set the number of trials

# Output the best hyperparameters
print("Best hyperparameters:", study.best_params)
print("Best validation accuracy:", study.best_value)

KeyboardInterrupt: 

In [10]:
def train(model, loader, optimizer, criterion, epochs=1):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        total = 0

        for keypoints, labels in tqdm(loader):
            optimizer.zero_grad()
            outputs = model(keypoints)  # Outputs shape should be [N, 1]
            loss = criterion(outputs, labels.view(-1, 1))  # Ensure labels are shaped [N, 1]
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            predictions = torch.sigmoid(outputs) > 0.5  # Convert to binary predictions
            correct += (predictions == labels.view(-1, 1)).sum().item()  # Ensure labels are shaped [N, 1]
            total += labels.size(0)

        avg_loss = total_loss / len(loader)
        avg_accuracy = correct / total
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}, Accuracy: {avg_accuracy:.4f}")

    return avg_accuracy, avg_loss

# Train the model
train_accuracy, train_loss = train(model, train_loader, optimizer, criterion, epochs=5)

 22%|██▏       | 16/72 [00:00<00:00, 158.42it/s]

LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.

 69%|██████▉   | 50/72 [00:00<00:00, 163.85it/s]

LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.

100%|██████████| 72/72 [00:00<00:00, 164.93it/s]


...

LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])


 85%|████████▍ | 61/72 [00:00<00:00, 151.97it/s]

LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])


100%|██████████| 72/72 [00:00<00:00, 144.44it/s]

LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([6, 1])
Epoch [5/5], Loss: 0.6077, Accuracy: 0.7084


## Testing + results

In [11]:
# Generate classification report and confusion matrix, then save them to a JSON file
def save_results_to_json(test_labels, test_predictions, train_accuracy, train_loss, test_accuracy, test_loss, output_path="classification_results.json"):
    # No need to convert test_labels to NumPy as it is already an array
    test_labels_np = test_labels  # Keep as is
    test_predictions_np = np.array(test_predictions)

    # Classification report and confusion matrix
    class_report = classification_report(test_labels_np, test_predictions_np, output_dict=True, zero_division=1)
    conf_matrix = confusion_matrix(test_labels_np, test_predictions_np).tolist()

    # Create a dictionary to store the results
    results = {
        "classification_report": class_report,
        "confusion_matrix": conf_matrix,
        "training_accuracy": train_accuracy,
        "training_loss": train_loss,
        "test_accuracy": test_accuracy,
        "test_loss": test_loss
    }

    # Save to JSON
    with open(output_path, "w") as f:
        json.dump(results, f, indent=4)


def test(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    predictions_list = []

    with torch.no_grad():
        for keypoints, labels in loader:
            outputs = model(keypoints)  # Outputs shape should be [N, 1]
            loss = criterion(outputs, labels.view(-1, 1))  # Ensure labels are shaped [N, 1]
            total_loss += loss.item()
            predictions = torch.sigmoid(outputs) > 0.5  # Convert to binary predictions
            predictions_list.extend(predictions.cpu().numpy())
            correct += (predictions == labels.view(-1, 1)).sum().item()  # Ensure labels are shaped [N, 1]
            total += labels.size(0)

    avg_loss = total_loss / len(loader)
    avg_accuracy = correct / total
    return avg_accuracy, avg_loss, predictions_list


# Test the model
test_accuracy, test_loss, predictions = test(model, test_loader, criterion)

# Save the results
save_results_to_json(test_labels, predictions, train_accuracy, train_loss, test_accuracy, test_loss)

LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([16, 1])
LSTM Output shape: torch.Size([1, 1])
